# 021 — Contextual Retrieval
**Advanced RAG Series** | Anthropic's 2024 technique for better chunk-level retrieval

Covers: Context prepending · Contextual BM25 · Contextual embeddings · Before/after comparison


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
import os
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(f"Missing: {path}\nRun python create_datasets.py from project root.")

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, rag_text])
print(f"✓ Loaded {len(all_text):,} chars across 4 files")


✓ Loaded 22,486 chars across 4 files


In [4]:
# ── 1. Standard chunking (baseline) ─────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
# Use a subset for faster demo
chunks = splitter.split_text(rag_text)
print(f"Total chunks: {len(chunks)}")
print("\n--- Chunk without context (loses surrounding info) ---")
print(chunks[5][:300])


Total chunks: 21

--- Chunk without context (loses surrounding info) ---
- Late chunking: embed full document, then split; preserves global context in chunk embeddings


In [5]:
# ── 2. Generate context for each chunk using LLM ────────────────────────
# Anthropic's approach: ask LLM "what is this chunk about in the full document?"

CONTEXT_PROMPT = (
    "Here is the full document:\n\n"
    "{document}\n\n"
    "Here is a specific chunk from that document:\n\n"
    "{chunk}\n\n"
    "Write a SHORT (1-2 sentence) context description that explains what this chunk "
    "is about within the full document. Be concise. Output ONLY the context, nothing else."
)

def generate_chunk_context(document: str, chunk: str) -> str:
    prompt = CONTEXT_PROMPT.format(document=document[:3000], chunk=chunk)
    response = llm.invoke(prompt)
    return response.content.strip()

# Demo on one chunk
sample_chunk = chunks[5]
context = generate_chunk_context(rag_text, sample_chunk)
print("Original chunk:")
print(sample_chunk[:200])
print("\nGenerated context:")
print(context)


Original chunk:
- Late chunking: embed full document, then split; preserves global context in chunk embeddings

Generated context:
This chunk discusses different strategies for chunking documents in the Retrieval-Augmented Generation (RAG) pipeline, focusing on the advantages of late chunking.


In [6]:
# ── 3. Build contextual chunks (prepend context to each chunk) ──────────
import time

# For the demo, process a subset (full corpus would cost many LLM calls)
demo_chunks = chunks[:15]
contextual_chunks = []

print("Generating context for 15 chunks...")
for i, chunk in enumerate(demo_chunks):
    ctx = generate_chunk_context(rag_text, chunk)
    contextual_chunk = f"{ctx}\n\n{chunk}"
    contextual_chunks.append(contextual_chunk)
    if (i+1) % 5 == 0:
        print(f"  {i+1}/15 done")
    time.sleep(0.2)  # rate limit

print("\n✓ Done. Sample contextual chunk:")
print(contextual_chunks[5][:400])


Generating context for 15 chunks...


  5/15 done


  10/15 done


  15/15 done

✓ Done. Sample contextual chunk:
This chunk discusses different strategies for chunking documents in the Retrieval-Augmented Generation (RAG) pipeline, focusing on the advantages of late chunking.

- Late chunking: embed full document, then split; preserves global context in chunk embeddings


In [7]:
# ── 4. BM25 retrieval: standard vs contextual ───────────────────────────
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.sub(r'[^a-z0-9\s]', ' ', text.lower()).split()

# Build two BM25 indexes
bm25_standard    = BM25Okapi([tokenize(c) for c in demo_chunks])
bm25_contextual  = BM25Okapi([tokenize(c) for c in contextual_chunks])

def retrieve(bm25_index, source_chunks, query, top_k=3):
    scores = bm25_index.get_scores(tokenize(query))
    top_idx = scores.argsort()[-top_k:][::-1]
    return [(source_chunks[i], float(scores[i])) for i in top_idx]

query = "how does RAG reduce hallucination"
std_results = retrieve(bm25_standard, demo_chunks, query)
ctx_results = retrieve(bm25_contextual, contextual_chunks, query)

print(f"Query: {query!r}\n")
print("STANDARD top-1 (score={:.3f}):".format(std_results[0][1]))
print(std_results[0][0][:250])
print("\nCONTEXTUAL top-1 (score={:.3f}):".format(ctx_results[0][1]))
print(ctx_results[0][0][:250])


Query: 'how does RAG reduce hallucination'

STANDARD top-1 (score=2.445):
RAG was introduced by Lewis et al. (2020) as a way to give language models access to
up-to-date information and verifiable sources, addressing key limitations of parametric
(weights-only) language models: knowledge staleness, hallucination, and lack 

CONTEXTUAL top-1 (score=2.888):
This chunk describes the introduction of Retrieval-Augmented Generation (RAG) and its purpose in addressing limitations of parametric language models.

RAG was introduced by Lewis et al. (2020) as a way to give language models access to
up-to-date in


In [8]:
# ── 5. Contextual embeddings (with HuggingFace) ─────────────────────────
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np, os

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

std_vecs = np.array(embeddings.embed_documents(demo_chunks))
ctx_vecs = np.array(embeddings.embed_documents(contextual_chunks))

def cosine_sim(a, b): return float(np.dot(a, b))

q_vec = np.array(embeddings.embed_query(query))
std_scores = [cosine_sim(q_vec, v) for v in std_vecs]
ctx_scores = [cosine_sim(q_vec, v) for v in ctx_vecs]

best_std = max(std_scores)
best_ctx = max(ctx_scores)
print(f"Query: {query!r}")
print(f"Best standard embedding score : {best_std:.4f}")
print(f"Best contextual embedding score: {best_ctx:.4f}")
print(f"Improvement: {(best_ctx - best_std) / best_std * 100:+.1f}%")


Query: 'how does RAG reduce hallucination'
Best standard embedding score : 0.3395
Best contextual embedding score: 0.2409
Improvement: -29.1%


In [9]:
# ── 6. Summary: Contextual Retrieval key points ─────────────────────────
print("CONTEXTUAL RETRIEVAL — KEY POINTS")
print("=" * 45)
print()
print("Problem it solves:")
print("  Chunks lose context when split from the full document.")
print("  e.g. 'it reduces hallucination' — what is 'it'?")
print()
print("Solution:")
print("  For each chunk, use LLM to generate a 1-2 sentence context")
print("  description. Prepend it to the chunk before indexing.")
print()
print("Result (per Anthropic 2024 report):")
print("  49% reduction in retrieval failures (BM25)")
print("  35% reduction in retrieval failures (embeddings)")
print()
print("Cost:")
print("  One LLM call per chunk at index-build time.")
print("  Use prompt caching to reduce cost ~80% for large corpora.")
print("  Zero extra cost at query time.")


CONTEXTUAL RETRIEVAL — KEY POINTS

Problem it solves:
  Chunks lose context when split from the full document.
  e.g. 'it reduces hallucination' — what is 'it'?

Solution:
  For each chunk, use LLM to generate a 1-2 sentence context
  description. Prepend it to the chunk before indexing.

Result (per Anthropic 2024 report):
  49% reduction in retrieval failures (BM25)
  35% reduction in retrieval failures (embeddings)

Cost:
  One LLM call per chunk at index-build time.
  Use prompt caching to reduce cost ~80% for large corpora.
  Zero extra cost at query time.
